# **Model Predictive Control (MPC) Closed-Loop Implementation & Validation**

Now that our predictive RC model and Extended Kalman Filter (EKF) observer are fully validated, we are ready to close the loop. This notebook implements the final **Model Predictive Control (MPC)** framework to achieve optimal, real-time climate control across our 5-zone PTAC system.

Unlike traditional classical controllers like PID, MPC operates by looking into the future. At each simulation timestep, the controller takes the current state estimate from our EKF—including hidden variables like wall mass temperatures ($T_m$) and unmeasured disturbances ($d_T, d_W$)—and solves a constrained optimization problem over a moving future time horizon. It then calculates the optimal trajectory for our HVAC hardware variables (supply airflow, temperature setpoints, and humidifier commands) and dispatches the very first step of that plan back to EnergyPlus.

### 🎯 Key Objectives:

* 🔋 **Minimize Energy Consumption:** Explicitly reduce the electrical runtime power of supply fans, cooling coils, and heating elements.
* 🌡️ **Maintain Strict Comfort Bounds:** Ensure room air temperatures ($T_{in}$), relative humidity levels ($RH$), and carbon dioxide concentrations ($CO_2$) stay safely within acceptable occupant limits.
* 🛡️ **Respect Physical Hardware Constraints:** Guarantee that the optimized control commands never exceed real-world equipment capacities, such as maximum fan mass flow rates.


### Installation And Setup Environment

In [ ]:
# @title Setup Simulation Environment
# 1. Download the raw simulator.py file directly from your GitHub repository
!wget -q -O simulator.py https://raw.githubusercontent.com/janithcyapa/DHCA-Framework/refs/heads/main/Simulation/simulator.py

# 2. Import it exactly like a standard Python library
from simulator import *
print("✅ Simulator library successfully downloaded and loaded!")
SetupSimulationEnv()

### MPC Implementation

### Simualtion

In [ ]:
# @title Run Complete Simualtion
sim = SetSimulationModel(verbose=0)
ModifySimulationModel(sim)
SetupStateLogger(sim)
SetupOccupancyInjector(sim)
SetupActuatorController(sim)
SetupEKFEstimator(sim,'SPACE1-1', False )
RunSimulation(sim, num_days=31, rate=60)
df_log,csv  = ExtractSimulationData(sim, "EKF_validation_sim.csv")

In [ ]:
# @title Plot Data
import plotly.io as pio
pio.renderers.default = "vscode"

layout_config = {
    "global_title": "MPC Closed-Loop Tracking & Hardware Validation: {target_zone}",
    "height": 2200,  # Increased height to fit 7 subplots cleanly
    "template": "plotly_dark",
    "vertical_spacing": 0.03,
    "subplots": [
        {
            "title": "1. System Temperatures (°C)",
            "y_title": "Temp (°C)",
            "group_id": "temp_air",
            "traces": [
                {"col": "T_out", "name": "Outdoor (Ambient)", "color": "white", "dash": "dash"},
                {"col": "{target_zone}_T_in", "name": "Room (Actual E+)", "color": "#00e5ff", "width": 2},
                {"col": "{target_zone}_T_in_pred", "name": "Room (Estimated)", "color": "#ff00ff", "dash": "dot", "width": 2},
                {"col": "{target_zone}_T_supply", "name": "Supply (Actual)", "color": "#ffaa00", "width": 2},
                {"col": "{target_zone}_Cmd_Temp_Setpt", "name": "Supply (Cmd Setpoint)", "color": "yellow", "dash": "dash"}
            ]
        },
        {
            "title": "2. Humidity Ratio (kg/kg)",
            "y_title": "W (kg/kg)",
            "group_id": "humidity",
            "traces": [
                {"col": "W_out", "name": "Outdoor (Ambient)", "color": "white", "dash": "dash"},
                {"col": "{target_zone}_W_in", "name": "Room (Actual E+)", "color": "#00e5ff", "width": 2},
                {"col": "{target_zone}_W_in_pred", "name": "Room (Estimated)", "color": "#ff00ff", "dash": "dot", "width": 2},
                {"col": "{target_zone}_W_supply", "name": "Supply (Actual)", "color": "#ffaa00", "width": 2}
            ]
        },
        {
            "title": "3. CO2 Concentration (ppm)",
            "y_title": "CO2 (ppm)",
            "group_id": "co2",
            "traces": [
                {"col": "CO2_out", "name": "Outdoor (Ambient)", "color": "white", "dash": "dash"},
                {"col": "{target_zone}_CO2_in", "name": "Room (Actual E+)", "color": "#00e5ff", "width": 2},
                {"col": "{target_zone}_C_in_pred", "name": "Room (Estimated)", "color": "#ff00ff", "dash": "dot", "width": 2},
                {"col": "{target_zone}_C_supply", "name": "Supply (Actual)", "color": "#ffaa00", "width": 2}
            ]
        },
        {
            "title": "4. Actuator: Fan Air Flow Tracking",
            "y_title": "Mass Flow (kg/s)",
            "group_id": "act_fan",
            "traces": [
                {"col": "{target_zone}_Cmd_Fan_Flow", "name": "Commanded Flow", "color": "yellow", "dash": "dash", "width": 2},
                {"col": "{target_zone}_m_dot", "name": "Actual Flow (E+)", "color": "#00f2ff", "width": 2, "fill": "tozeroy"}
            ]
        },
        {
            "title": "5. Actuator: Cooler Tracking",
            "y_title": "Temp (°C)",
            "secondary_y": True,
            "y_title_secondary": "Cooling Power (W)",
            "group_id": "act_cool",
            "traces": [
                {"col": "{target_zone}_Cmd_Clg_Setpt", "name": "Cooling Setpoint (Cmd)", "color": "blue", "dash": "dash", "width": 2},
                {"col": "{target_zone}_T_in", "name": "Room Temp (Actual)", "color": "white", "width": 2},
                {"col": "{target_zone}_Cool_Rate", "name": "Actual Cooling Power", "color": "cyan", "width": 2, "sec_y": True, "fill": "tozeroy"}
            ]
        },
        {
            "title": "6. Actuator: Heater Tracking",
            "y_title": "Temp (°C)",
            "secondary_y": True,
            "y_title_secondary": "Heating Power (W)",
            "group_id": "act_heat",
            "traces": [
                {"col": "{target_zone}_Cmd_Htg_Setpt", "name": "Heating Setpoint (Cmd)", "color": "red", "dash": "dash", "width": 2},
                {"col": "{target_zone}_T_in", "name": "Room Temp (Actual)", "color": "white", "width": 2},
                {"col": "{target_zone}_Heat_Rate", "name": "Actual Heating Power", "color": "orange", "width": 2, "sec_y": True, "fill": "tozeroy"}
            ]
        },
        {
            "title": "7. Actuator: Humidifier Tracking",
            "y_title": "Latent Power (W)",
            "group_id": "act_hum",
            "traces": [
                {"col": "{target_zone}_Hum_Cmd_Rate", "name": "Commanded Power", "color": "yellow", "dash": "dash", "width": 2},
                {"col": "{target_zone}_Hum_Actual_Rate", "name": "Actual Power (E+)", "color": "#B200FF", "width": 2, "fill": "tozeroy"},
                {"col": "{target_zone}_Hum_Rate", "name": "Actual Power 2(E+)", "color": "#00FF91", "width": 2, "fill": "tozeroy"},

            ]
        }

        
    ]
}

fig = GenerateSimulationPlots(df_log, target_zone="SPACE1-1", layout_config=layout_config)

## Setup Simulator

In [12]:
# @title zone_mpc
def zone_mpc(self, state):
    """Multi-zone Model Predictive Control (logs stored in self.mpc_logs)."""
    if not self.exchange.api_data_fully_ready(state) or self.exchange.warmup_flag(state):
        return

    T_ref = 16.0   # °C

    # 1. Retrieve all active EKF zones
    if not hasattr(self, 'zones_ekf') or not self.zones_ekf:
        return

    # Ensure the MPC log container exists
    if not hasattr(self, 'mpc_logs'):
        self.mpc_logs = {}
        all_params = self.get_zone_thermal_parameters()
        self.ekf_zone_list = list(all_params.keys())
        print(f"[MPC] Will handle zones: {self.ekf_zone_list}")

    if not hasattr(self, 'vav_targets'):
        self.vav_targets = {}

    day   = self.exchange.day_of_year(state)
    time  = self.exchange.current_time(state)
    dt_hr = self.exchange.system_time_step(state) or self.exchange.zone_time_step(state)
    dt    = dt_hr * 3600.0           # seconds
    if dt <= 0:
        return

    # Physical constants
    rho_air, cp_air = 1.204, 1006.0
    q_person   = 100.0               # W/person
    g_w_person = 5e-5                # kg/s·person
    g_co2_person = 1e-5              # m³ CO₂/s·person (at room conditions)

    # ASHRAE‑based comfort limits (internal units)
    CO2_max_frac = 1000 * 1e-6       # 1000 ppm → volumetric fraction
    W_max_kgkg   = 0.012             # approx. 70 % RH at 22 °C

    # Actuator limits
    u_max = 0.5                      # realistic VAV maximum (m³/s)
    delta_u_max = 0.1                # max change per 5‑min step (m³/s)

    # MPC horizon and weights
    Np, nx, nu = 10, 4, 1
    Q_diag = [100.0, 1.0, 1e-4, 1e-4]   # T_in, T_m, C_in, W_in
    R_val = 0.1                          # penalise large u deviations
    R_delta_val = 1.0                    # slew rate penalty
    rho_c_soft = 1e6                     # slack penalty for CO₂
    rho_w_soft = 1e6                     # slack penalty for humidity

    # Loop over all zones with an EKF
    for zone_id, z_ekf in self.zones_ekf.items():
        if z_ekf.X_est is None:
            continue

        # ----- Extract EKF estimates (all in SI units) -----
        T_in_e, T_m_e, W_in_e, C_in_frac, d_T_e, d_W_e, N_occ_e = z_ekf.X_est
        x_k = np.array([T_in_e, T_m_e, C_in_frac, W_in_e])

        # ----- Boundary conditions (convert supply CO₂ from ppm → fraction) -----
        T_out = self.exchange.get_variable_value(state, z_ekf.handles["T_out"])
        T_s   = self.exchange.get_variable_value(state, z_ekf.handles["T_s"])
        W_s   = self.exchange.get_variable_value(state, z_ekf.handles["W_s"])
        C_s_ppm = self.exchange.get_variable_value(state, z_ekf.handles["C_s"])
        C_s_frac = C_s_ppm * 1e-6

        # Previous control input (for re‑linearisation and slew rate)
        u_op = getattr(z_ekf, 'u_prev', 0.05)

        # ----- Linearisation (continuous state matrix A_c) -----
        inv_R_ext = 1.0 / z_ekf.R_env_ext if z_ekf.R_env_ext > 0 else 0.0
        inv_R_int = 1.0 / z_ekf.R_int if z_ekf.R_int > 0 else 0.0
        inv_R_adj_total = getattr(z_ekf, 'inv_R_adj_sum', 0.0)

        Ac = np.zeros((4, 4))
        Ac[0, 0] = (-inv_R_ext - inv_R_int - inv_R_adj_total - (rho_air * cp_air * u_op)) / z_ekf.C_air
        Ac[0, 1] = inv_R_int / z_ekf.C_air
        Ac[1, 0] = inv_R_int / z_ekf.C_mass
        Ac[1, 1] = -inv_R_int / z_ekf.C_mass
        Ac[2, 2] = -u_op / z_ekf.V_room
        Ac[3, 3] = -(rho_air * u_op) / z_ekf.M_air

        Bc = np.zeros((4, 1))
        # Guard against zero temperature difference
        delta_T = T_s - T_in_e
        if abs(delta_T) < 0.5:   # too small → uncontrollable, keep previous
            u_opt = u_op
            m_dot_opt = float(u_opt * rho_air)
            h_act = self.exchange.get_actuator_handle(state, "System Node Setpoint",
                         "Mass Flow Rate Setpoint", f"{zone_id} In Node")
            if h_act != -1:
                self.exchange.set_actuator_value(state, h_act, m_dot_opt)

            # Log to new container
            if zone_id not in self.mpc_logs:
                self.mpc_logs[zone_id] = []
            self.mpc_logs[zone_id].append({
                "day": day, "hour": time, "T_in_est": T_in_e,
                "T_m_est": T_m_e, "W_in_est": W_in_e, "C_in_est": C_in_frac*1e6,
                "N_occ_est": N_occ_e, "T_ref": T_ref, "T_ss": T_in_e,
                "u_ss": u_op, "u_opt_m3s": u_opt
            })
            continue


        Bc[0, 0] = (rho_air * cp_air * delta_T) / z_ekf.C_air
        Bc[2, 0] = (C_s_frac - C_in_frac) / z_ekf.V_room
        Bc[3, 0] = (rho_air * (W_s - W_in_e)) / z_ekf.M_air

        # ----- Drift vector c_c (continuous) -----
        t_adj_flux = 0.0
        for adj in z_ekf.adj_zones:
            t_adj = self.exchange.get_variable_value(state, adj["handle_T_in"])
            t_adj_flux += t_adj / float(adj["R_env"])

        f_op = np.zeros((4, 1))
        f_op[0, 0] = ((T_out - T_in_e) * inv_R_ext +
                      (T_m_e - T_in_e) * inv_R_int +
                      (t_adj_flux - T_in_e * inv_R_adj_total) +
                      (rho_air * u_op * cp_air * (T_s - T_in_e)) +
                      (N_occ_e * q_person) + d_T_e) / z_ekf.C_air
        f_op[1, 0] = ((T_in_e - T_m_e) * inv_R_int) / z_ekf.C_mass
        f_op[2, 0] = (u_op * (C_s_frac - C_in_frac) + N_occ_e * g_co2_person) / z_ekf.V_room
        f_op[3, 0] = (rho_air * u_op * (W_s - W_in_e) + d_W_e) / z_ekf.M_air

        cc = f_op - np.dot(Ac, x_k.reshape(4,1)) - (Bc * u_op)

        # ----- Exact discretisation (augmented matrix exponential) -----
        M_aug = np.zeros((6, 6))
        M_aug[0:4, 0:4] = Ac
        M_aug[0:4, 4:5] = Bc
        M_aug[0:4, 5:6] = cc
        Md = la.expm(M_aug * dt)

        Ad = Md[0:4, 0:4]
        Bd = Md[0:4, 4:5]
        cd = Md[0:4, 5:6].flatten()   # affine term in discrete time

        # ----- Target selector (thermal subsystem only) -----

        Ad_th = Ad[0:2, 0:2]
        Bd_th = Bd[0:2, :]
        cd_th = cd[0:2]

        top_block = np.hstack([np.eye(2) - Ad_th, -Bd_th])
        bottom_block = np.array([[1.0, 0.0, 0.0]])   # track T_in
        M_ss = np.vstack([top_block, bottom_block])
        rhs_ss = np.concatenate([cd_th, [T_ref]])

        try:
            ss_res = np.linalg.solve(M_ss, rhs_ss)
            x_ss = np.array([ss_res[0], ss_res[1], C_in_frac, W_in_e])   # hold CO₂/W at current
            u_ss = ss_res[2]
        except np.linalg.LinAlgError:
            # Fallback: hold current state and previous control
            x_ss = x_k.copy()
            u_ss = u_op

        # ----- Build OSQP problem (absolute variables, slacks) -----
        # Decision vector: z = [x0, ..., x_Np, u0, ..., u_{Np-1}, eps_c0, ..., eps_w{Np-1}]
        # Total length: (Np+1)*nx + Np*nu + Np*2
        n_states = (Np+1)*nx + Np*nu + Np*2

        # Hessian
        Q = sparse.diags(Q_diag, format='csc')
        R = sparse.diags([R_val], format='csc')
        P = sparse.block_diag([
            sparse.kron(sparse.eye(Np+1), Q),
            sparse.kron(sparse.eye(Np), R),
            sparse.diags([rho_c_soft]*Np + [rho_w_soft]*Np)
        ], format='csc')

        # Linear gradient (to shift cost towards x_ss, u_ss, zero slacks)
        z_ss = np.hstack([
            np.tile(x_ss, Np+1),
            np.tile(u_ss, Np),
            np.zeros(Np),   # epsilon_c target = 0
            np.zeros(Np)    # epsilon_w target = 0
        ])
        q = -P @ z_ss

        # Constraints
        Ax_dyn = sparse.kron(sparse.eye(Np+1, Np+1), sparse.eye(nx)) - \
                 sparse.kron(sparse.diags([1]*Np, offsets=[-1], shape=(Np+1, Np+1)), Ad)
        Bu_dyn = sparse.kron(sparse.vstack([
            sparse.csc_matrix((1, Np)),
            sparse.eye(Np)
        ]), -Bd)
        A_eq = sparse.hstack([Ax_dyn, Bu_dyn, sparse.csc_matrix((Np*nx, 2*Np))], format='csc')
        l_eq = np.tile(cd, Np)
        u_eq = l_eq

        # Initial condition
        A0 = sparse.hstack([sparse.eye(nx),
                             sparse.csc_matrix((nx, Np*nx + Np*nu + 2*Np))], format='csc')
        A_eq = sparse.vstack([A0, A_eq])
        l_eq = np.hstack([x_k, l_eq])
        u_eq = np.hstack([x_k, u_eq])

        # Actuator limits
        A_u = sparse.hstack([
            sparse.csc_matrix((Np*nu, (Np+1)*nx)),
            sparse.eye(Np*nu),
            sparse.csc_matrix((Np*nu, 2*Np))
        ], format='csc')
        l_u = np.zeros(Np*nu)
        u_u = np.ones(Np*nu) * u_max

        # Slew-rate limits
        A_slew = sparse.hstack([
            sparse.csc_matrix((Np, (Np+1)*nx)),
            sparse.diags([-1, 1], offsets=[-1, 0], shape=(Np, Np)),
            sparse.csc_matrix((Np, 2*Np))
        ], format='csc')
        l_slew = np.ones(Np) * (-delta_u_max)
        u_slew = np.ones(Np) * ( delta_u_max)
        l_slew[0] += u_op
        u_slew[0] += u_op

        # Soft constraints on CO₂ and humidity
        A_co2_full = sparse.lil_matrix((Np, n_states))
        for i in range(Np):
            A_co2_full[i, i*nx + 2] = 1.0
        A_co2_full[:, (Np+1)*nx + Np*nu : (Np+1)*nx + Np*nu + Np] = -sparse.eye(Np)
        A_co2_full = A_co2_full.tocsc()

        A_w = sparse.lil_matrix((Np, n_states))
        for i in range(Np):
            A_w[i, i*nx + 3] = 1.0
        A_w[:, (Np+1)*nx + Np*nu + Np : (Np+1)*nx + Np*nu + 2*Np] = -sparse.eye(Np)
        A_w = A_w.tocsc()

        A_ineq = sparse.vstack([
            A_u,
            A_slew,
            A_co2_full,
            A_w
        ], format='csc')

        l_ineq = np.hstack([
            l_u,
            l_slew,
            np.ones(Np) * (-np.inf),
            np.ones(Np) * (-np.inf)
        ])
        u_ineq = np.hstack([
            u_u,
            u_slew,
            np.ones(Np) * CO2_max_frac,
            np.ones(Np) * W_max_kgkg
        ])

        A_slack_nonneg = sparse.hstack([
            sparse.csc_matrix((2*Np, (Np+1)*nx + Np*nu)),
            sparse.eye(2*Np)
        ])
        A_ineq = sparse.vstack([A_ineq, A_slack_nonneg], format='csc')
        l_ineq = np.hstack([l_ineq, np.zeros(2*Np)])
        u_ineq = np.hstack([u_ineq, np.ones(2*Np) * np.inf])

        # Add slew-rate penalty to Hessian
        D = sparse.eye(Np) - sparse.diags([1]* (Np-1), offsets=[-1], shape=(Np, Np))
        R_delta_mat = sparse.diags([R_delta_val]*Np, format='csc')
        P_blocks = [
            sparse.kron(sparse.eye(Np+1), Q),
            sparse.kron(sparse.eye(Np), R) + D.T @ R_delta_mat @ D,
            sparse.diags([rho_c_soft]*Np + [rho_w_soft]*Np)
        ]
        P = sparse.block_diag(P_blocks, format='csc')
        q = -P @ z_ss

        # ----- OSQP Setup / Update -----
        mpc_attr = f"mpc_{zone_id}"
        if not hasattr(self, mpc_attr):
            setattr(self, mpc_attr, {
                "prob": osqp.OSQP(),
                "initialized": False
            })
        mpc_data = getattr(self, mpc_attr)

        if not mpc_data["initialized"]:
            mpc_data["prob"].setup(P, q,
                                   sparse.vstack([A_eq, A_ineq]),
                                   np.hstack([l_eq, l_ineq]),
                                   np.hstack([u_eq, u_ineq]),
                                   verbose=False, warm_start=True)
            mpc_data["initialized"] = True
        else:
            mpc_data["prob"].update(q=q, l=np.hstack([l_eq, l_ineq]),
                                    u=np.hstack([u_eq, u_ineq]),
                                    Px=None)

        res = mpc_data["prob"].solve()

        if res.info.status_val == 1:
            u_opt = res.x[(Np+1)*nx]
        else:
            print(f"[MPC {zone_id}] Solver issue: {res.info.status}. Using u_ss.")
            u_opt = u_ss

        u_opt = np.clip(u_opt, 0.0, u_max)
        z_ekf.u_prev = u_opt

        # ----- Actuation -----
        m_dot_opt = float(u_opt * rho_air)
        self.vav_targets[zone_id] = m_dot_opt


        # ----- Logging (to separate container) -----
        if zone_id not in self.mpc_logs:
                self.mpc_logs[zone_id] = []
        self.mpc_logs[zone_id].append({
            "day": day, "hour": time, "T_in_est": T_in_e,
            "T_m_est": T_m_e, "W_in_est": W_in_e, "C_in_est": C_in_frac*1e6,
            "N_occ_est": N_occ_e, "T_ref": T_ref, "T_ss": T_in_e,
            "u_ss": u_op, "u_opt_m3s": u_opt
        })
        continue

        if int(time*60) % 60 == 0:
            print(f"[{zone_id} H:{time:.1f}] T:{T_in_e:.1f}°C T_ss:{x_ss[0]:.1f}°C "
                  f"u_ss:{u_ss:.3f} m³/s u_opt:{u_opt:.3f} m³/s "
                  f"CO2:{C_in_frac*1e6:.0f} ppm")

sim.zone_mpc = types.MethodType(zone_mpc, sim)
sim.register_handlers("before_hvac", [{"method_name": "zone_mpc"}])

['zone_mpc']